# SPOD to Mapping Excel for Geberit
- Prerequisites: 
  - Anaconda packages: `xlsxwriter` pandas, openpyxl, seaborn`


## Result

Excel sheet containing:

Sheet with all Datapoints (Systems, Tables and Columns) mapped against the Information Model (Entity, Attribute)
Table covering the overview sheet

Analog dev_x_mapping

Optional:
Sheet per System - IM containing sample data

## Structure

1. Define mapping between SPOD (json) and columns in the resulting Excel sheet

## Configuration
The following parameters has to be definded when running as regular python script

In [ ]:
LIBRARY = '../../pythonWork/pythonSource'
DESTINATION = 'mapping.xlsx'
MODEL_SOURCE = '/Users/bue/projects/geberit/DEAP/DB/IM_GEBERIT.json'

## Check prerequisites

In [ ]:
import sys
import logging
import os
import json
import yaml
from pathlib import Path

In [ ]:
spod_file = Path(MODEL_SOURCE)
assert spod_file.is_file(), f"Cannot find SPOD file '{spod_file.resolve()}'"

with open(spod_file, 'r') as src:
    spod = json.load(src)
assert spod['model'] is not None
print(f"Loaded SPOD containing {spod['model']} from '{spod_file.resolve()}'")

In [ ]:
print(f"Loaded {spod_file.resolve()}\n{spod['model']}\nVersion {spod['_imprint_']}")
print(f"Languages: {list(spod['languages'].keys())}")
mapdict = {}
for entry in ['entities', 'attributes', 'systems', 'columns']:
    print(f"- {entry}: {len(spod[entry])}")

## Initialize logging

In [ ]:
import logging
from logging import handlers
from datetime import datetime

stamp = datetime.now()
run_stamp = stamp.strftime("%Y-%m-%d-%H-%M-%S")

os.makedirs('log', exist_ok=True)
logfile = f'log/sharepoint-list-sync-{run_stamp}.log'

handler = handlers.RotatingFileHandler(logfile, maxBytes=(1024 * 1024 * 10), backupCount=10)
handler.setLevel(logging.DEBUG)

formatter = logging.Formatter("%(asctime)s [%(threadName)s] - %(name)s - %(levelname)s - %(message)s")
handler.setFormatter(formatter)

console_log_handler = logging.StreamHandler()
console_formatter = logging.Formatter("%(levelname)s - %(message)s")
console_log_handler.setFormatter(console_formatter)
console_log_handler.setLevel(logging.INFO)

logger = logging.getLogger()
logger.setLevel(logging.DEBUG)
logger.addHandler(handler)
logger.addHandler(console_log_handler)

urlliblogger = logging.getLogger('urllib3.connectionpool')
urlliblogger.setLevel(logging.DEBUG)

## Use the fyayc SPOD library

In [ ]:
toolpath = Path(LIBRARY)
assert toolpath.is_dir(), f"{toolpath.reslove()} is not a directory. The constant 'LIBRARY' must point to the library. Default = 'pythonWork/pythonSource'."
sys.path.insert(0, str(toolpath))

from PUBLISH_MODEL.excel.mapping_publisher import generate
from SSOT_infra.translator import Translator

## Translation shortcut tr

In [ ]:
translator = Translator('de')

## Define Systems order

In [ ]:
dm_sap_key, _ = next(filter(lambda kv: kv[1]['name'] == 'DM_SAP', spod['systems'].items()))
dm_step_key, _ = next(filter(lambda kv: kv[1]['name'] == 'DM_STEP', spod['systems'].items()))

In [ ]:
from collections import OrderedDict
systems_dict = OrderedDict(spod['systems'])
systems_dict.move_to_end(dm_sap_key, last=False) # Move to front
systems_dict.move_to_end(dm_step_key, last=False) # Move to front
spod['systems'] = systems_dict

## List structure definition

In [ ]:
columns_mapped = {}
system_index = {}

In [ ]:
headings_im = ["FQN", "EID", "AID", "Entity (EN) ", "Attribute (EN)", "Entity (DE) ", "Attribute (DE)", "Entity (FR) ", "Attribute (FR)", "Examples", "Description", "#"]

In [ ]:
len(headings_im)

In [ ]:
def emit_system_columns(spod: dict) -> [str]:
    result = []
    for key, system in spod['systems'].items():
        system_index[key] = len(headings_im) + len(result)
        result.append(key + ':ID')
        result.append(key + ':FQN')
        result.append(system['name'])
    return result

In [ ]:
systems = emit_system_columns(spod)
systems

In [ ]:
def export_column(key: str, column: dict) -> []:
    result = [
                column['interface_col_id'],
                column['interface-id+'] + '.' + column['table-id'] + '.' + key,
                column['name'],
            ]
    return result
    
def firsthit(spod: dict, attribute_key: str, system_key: str) -> []:
    for ckey, column in spod['columns'].items():
        if ckey not in columns_mapped and system_key == column['interface-id+'] and attribute_key in column['attributesmapped']:
            result = export_column(ckey, column)
            columns_mapped[ckey] = attribute_key
            return result
    return [ '', '', '' ]

In [ ]:
def emit_row(spod: dict, attribute_key: str, attribute: dict, translator: Translator) -> []:
    enti_key = attribute['entity']
    entity = spod['entities'].get(enti_key)
    assert entity is not None, f"Missing entity {enti_key}"
    result = [
        enti_key + ':' + attribute_key,
        enti_key,
        attribute_key,
        translator.tr(entity['name'], 'en'),
        translator.tr(attribute['name'], 'en'),
        translator.tr(entity['name'], 'de'),
        translator.tr(attribute['name'], 'de'),
        translator.tr(entity['name'], 'fr'),
        translator.tr(attribute['name'], 'fr'),
        ', '.join(translator.tr(attribute.get('examples'), 'en')),
        translator.tr(attribute['descr'], 'en'),
        len(attribute['columnsmapped+']),
    ]

    for skey in spod['systems'].keys():
        mapping = firsthit(spod, attribute_key, skey)
        result = result + mapping

    return result

In [ ]:
headings = headings_im + systems
f"Columns ({len(headings)}): {', '.join(headings)}"

# Create data table (content)

In [ ]:
data_table = [ emit_row(spod, key, attribute, translator) for key, attribute in spod['attributes'].items() ]

In [ ]:
len(data_table)

In [ ]:
f"Already mapped {len(columns_mapped)} columns of {len(spod['columns'])}"

In [ ]:
list(columns_mapped.items())[0:3]

## Append unmapped columns to the bottom

In [ ]:
def aux_row(spod: dict, key: str, column: dict, translator: Translator) -> []:
    mapped = column['attributesmapped']
    if len(mapped) > 0:
        attrkey = mapped[0]
        attr = spod['attributes'][attrkey]
        result = emit_row(spod, attrkey, attr, translator)
        result.extend( [ None ] * (len(headings) - len(result))  )
    else:
        result = [ None ] * len(headings)
    
    map_count = len(column['attributesmapped'])
    result[len(headings_im) - 1] = map_count

    index = system_index[column['interface-id+']]
    values = export_column(key, column)
    result[index + 0] = values[0]
    result[index + 1] = values[1]
    result[index + 2] = values[2]
    
    
    return result

In [ ]:
remainder = [ aux_row(spod, key, spod['columns'][key], translator) for key in filter(lambda key: key not in columns_mapped.keys(), spod['columns'].keys()) ]

In [ ]:
data_table = data_table + remainder

In [ ]:
### Sort by Attribute FQN
data_table.sort(key=lambda r: r[0] if r[0] is not None else '\uFFFF')

# xlsxwriter Approach

In [ ]:
import xlsxwriter

In [ ]:
dest = Path(DESTINATION)
xlsx_destination = dest
workbook = xlsxwriter.Workbook(xlsx_destination)

title_format = workbook.add_format({'bold': True, 'font_color': 'black', 'font_size': 20})

head_format = {'bold': True, 'bg_color': '#A0A0A0'}
column_head_format = workbook.add_format(head_format)

column_head_rotated_format = workbook.add_format(head_format)
column_head_rotated_format.set_rotation(90)

In [ ]:
## Summary is first sheet, but will be filled last
summary = workbook.add_worksheet('Summary')

In [ ]:
## Prepare Mapping sheet

In [ ]:
worksheet = workbook.add_worksheet('Mapping')

col = 0
for header in headings:
    worksheet.write(0, col, header)
    col += 1

row = 1
for entry in data_table:
    col = 0
    for item in entry:
        worksheet.write(row, col, item)
        col += 1
    row += 1

### Define Table

In [ ]:
table_column_headers = [ { 'header': name } for name in headings ]

In [ ]:
worksheet.add_table(0, 0, len(data_table) + 1, len(headings) - 1, { 
    'name': 'mapping',
    'banded_rows': True,
    'columns': table_column_headers,
})

### Styling

In [ ]:
def collapsed_system(system: dict) -> bool:
    return system['name'] not in ['DM_SAP', 'DM_STEP'] 

In [ ]:
# FQN width
worksheet.set_column(0, 0, 20)

# Hide EID, AID on the left
worksheet.set_column(1, 3, 10, None, { 'hidden': 1, })

# EID, AID
worksheet.set_column(3, 5, 20)

# Hide attribute name translations (DE, FR)
worksheet.set_column(5, 8, 40, None, { 'hidden': 1, })

base = len(headings_im)
index = 0
for system in spod['systems'].values():
    colnr = base + (index * 3)
    # Column name on system
    worksheet.set_column(colnr + 0, colnr + 2, 40)
    
    start_letter = xlsxwriter.utility.xl_col_to_name(colnr + 0)
    end_letter = xlsxwriter.utility.xl_col_to_name(colnr + 1)
    group = f'{start_letter}:{end_letter}'
    collapsed = collapsed_system(system)
    logging.debug(f"Group {group} is {'collapsed' if collapsed else 'visible'}")
    worksheet.set_column(colnr + 0, colnr + 1, 35, None, {'level': 1, 'hidden': True})
    worksheet.set_column(colnr + 2, colnr + 2, 35, None, {'collapsed': collapsed})
    
    index += 1

## Add one sheet per system

In [ ]:
def fill_worksheet(spod: dict, skey: str, system: str, sheet):
    row = 0
    hf = column_head_format
    sheet.write(row, 0, 'Table Key', hf)
    sheet.set_column(0, 0, 30, None, {'hidden': 1})
    sheet.write(row, 1, 'Table Name', hf)
    sheet.set_column(1, 1, 40, None)
    sheet.write(row, 2, 'Column Key', hf)
    sheet.set_column(2, 2, 30, None, {'hidden': 1})
    sheet.write(row, 3, 'Name', hf)
    sheet.set_column(3, 3, 70, None)
    sheet.write(row, 4, 'ID', hf)
    sheet.set_column(4, 5, 40, None)
    sheet.write(row, 5, 'Datatype', hf)
    sheet.write(row, 6, 'Mandatory', hf)
    sheet.write(row, 7, 'Description', hf)
    sheet.set_column(7, 7, 80, None)

    sheet.write(row, 8, 'Transformation Rule', hf)
    sheet.write(row, 9, 'Validation Rule', hf)
    sheet.write(row, 10, 'Error Handling', hf)
    
    sheet.write(row, 11, '|', column_head_format)
    sheet.set_column(11, 11, 1, None)
    sheet.write(row, 12, 'IM Attributes', hf)
    sheet.set_column(12, 12, 50, None)
    
    row += 1
    columns = sorted(list(spod['columns'].items()), key=lambda c: c[1]['table-name+'])
    for ckey, column in columns:
        if column['interface-id+'] == skey:
            sheet.write(row, 0, column['table-id'])
            sheet.write(row, 1, column['table-name+'])
            sheet.write(row, 2, ckey)
            sheet.write(row, 3, column['name'])
            sheet.write(row, 4, column['interface_col_id'])
            sheet.write(row, 5, column['datatype'])
            sheet.write(row, 6, column['mandatory'])
            sheet.write(row, 7, translator.tr(column['descr'], 'de'))
            
            sheet.write(row, 11, '|')
            sheet.write(row, 12, ', '.join(column['attributesmapped']))
            row += 1
            
    return row

In [ ]:
import re

worksheets = dict()

for key, system in spod['systems'].items():
    title = re.sub(r'[\:\[\]*?/\\]', '_', system['name'])
    length = min(25, len(title))
    t = key.replace('INTF','') + ' ' + title[:length]
    if t.lower() in worksheets.keys():
        t = key.replace('INTF','') + ' ' + title[max(0, len(title) - 25):]
    worksheet = workbook.add_worksheet(t)
    worksheets[t.lower()] = worksheet
    rows = fill_worksheet(spod, key, system, worksheet)
    print(f"{key}: {system['name']} -> {t} with {rows} rows")

## Summary sheet

In [ ]:
summary.write(0, 0, "Summary", title_format)

row = 2
summary.write(row, 0, 'Key', column_head_format)
summary.write(row, 1, 'Name', column_head_format)
summary.write(row, 2, 'Mapped', column_head_format)
summary.write(row, 3, 'Total', column_head_format)
summary.write(row, 4, 'Tables', column_head_format)
summary.write(row, 5, 'Sheet Link', column_head_format)

row = 3
for skey, system in spod['systems'].items():
    summary.write(row, 0, skey)
    summary.write(row, 1, system['name'])
    
    columns = list(filter(lambda c: c['interface-id+'] == skey, spod['columns'].values()))
    mapped = list(filter(lambda c: len(c['attributesmapped']) > 0, columns))
    summary.write(row, 2, len(mapped))
    summary.write(row, 3, len(columns))
    
    summary.write(row, 4, len(system['tables+']))
    
    _, ws = next(iter(filter(lambda t: skey[4:] in t[0], worksheets.items())))
    summary.write_url(row, 5, f"internal:'{ws.get_name()}'!A1", string=f'Sheet {skey[4:]}')
    
    row += 1

## Styling

In [ ]:
summary.set_column(0, 0, 20)
summary.set_column(1, 1, 60)
summary.set_column(2, 5, 15)

## Write Excel file

In [ ]:
version_file = Path(LIBRARY, 'versons.json')
if version_file.is_file():
    with open(version_file, 'r') as src:
        version = json.load(src)
else:
    version = { 'TOOLVERSION': '?.?' }

In [ ]:
workbook.set_properties({
    'title':    f"{spod['model']['name']}",
    'subject':  'mapping',
    'author':   f"Excel Mapping Publisher {version['TOOLVERSION']}",
#    'manager':  'D',
#    'company':  'of Wolves',
    'category': 'export',
    'keywords': 'Information Model, Data Models',
    'comments': f"generated with {version['TOOLVERSION']} from model {spod['_imprint_']['Modelversion']}",
    'status':   'Draft',
    'revision': spod['_imprint_'].get('git')
})

In [ ]:
workbook.close()
print(f"Wrote {xlsx_destination}")

# Visually verify

In [ ]:
import subprocess
r = subprocess.run(['qlmanage', '-x', '-p', xlsx_destination], shell=False) # capture_output=False, stderr=subprocess.DEVNULL)

In [ ]:
import pandas
excel_data_df = pandas.read_excel(DESTINATION, sheet_name='Mapping')

In [ ]:
from IPython.display import display, HTML
display(excel_data_df)